# 00 · Set up the project environment

Run this **once after cloning** (or any time you want to rebuild). It uses only Python's standard library, so it runs on any Python 3.9+.

**What it does**
1. Finds the newest suitable Python already installed on your machine
2. Creates a project-local virtual environment (`.venv/`)
3. Installs everything in `requirements.txt`
4. Registers a Jupyter kernel called **Python (EIA SEDS)**
5. Creates `.env` from `.env.example` and checks your API key is filled in

**What it can't do:** upgrade the Python that is running this notebook. A notebook can't replace its own interpreter. It *can* build a new environment from a newer Python that is already installed, and if there isn't one, it tells you how to install one.

Run the cells top to bottom. Settings are in the next cell.

In [ ]:
# ---- Settings (edit here) ----
MIN_PYTHON = (3, 10)            # oldest Python this project supports
RECOMMENDED_PYTHON = (3, 12)    # the version we want people on
VENV_DIR = ".venv"
KERNEL_NAME = "eia-seds"
KERNEL_DISPLAY_NAME = "Python (EIA SEDS)"
RECREATE_VENV = False           # True = move the old environment aside and rebuild with the newest Python found
ALLOW_OLD_PYTHON = False        # True = continue even if the only Python found is older than MIN_PYTHON

In [ ]:
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

IS_WINDOWS = platform.system() == "Windows"


def find_repo_root(start=None):
    """Walk up from the notebook's folder until we find requirements.txt."""
    here = Path(start or Path.cwd()).resolve()
    for folder in [here] + list(here.parents):
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Could not find requirements.txt. Open this notebook from inside the project folder.")


ROOT = find_repo_root()


def run(cmd, show=True, tail=None):
    """Run a command, print its output, and stop with a clear error if it fails."""
    cmd = [str(c) for c in cmd]
    shown = [c if "\n" not in c else "<code>" for c in cmd]
    print("$", " ".join(shown))
    result = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
    if show and result.stdout.strip():
        lines = result.stdout.strip().splitlines()
        print("\n".join(lines[-tail:] if tail else lines))
    if result.returncode != 0:
        print(result.stderr.strip() or result.stdout.strip())
        raise RuntimeError("Command failed (exit code %d)." % result.returncode)
    return result


print("Project root:", ROOT)
print("This notebook is running on Python", platform.python_version(), "at", sys.executable)

## 1 · Find a suitable Python

In [ ]:
def interpreter_version(cmd):
    """Return (major, minor) for an interpreter command, or None if it doesn't run."""
    try:
        out = subprocess.run(
            list(cmd) + ["-c", "import sys; print(sys.version_info[0], sys.version_info[1])"],
            capture_output=True, text=True, timeout=20,
        )
        major, minor = out.stdout.split()
        return (int(major), int(minor))
    except Exception:
        return None


def candidate_interpreters():
    candidates = [[sys.executable]]
    for minor in range(15, MIN_PYTHON[1] - 1, -1):
        found = shutil.which("python3.%d" % minor)
        if found:
            candidates.append([found])
        if IS_WINDOWS and shutil.which("py"):
            candidates.append(["py", "-3.%d" % minor])
    return candidates


best = None
for cmd in candidate_interpreters():
    version = interpreter_version(cmd)
    if version and (best is None or version > best[0]):
        best = (version, cmd)

INSTALL_HELP = (
    "Install Python %d.%d, then restart VS Code (or Jupyter) and run this notebook again:\n"
    "  macOS:   brew install python@%d.%d   (or the installer from https://www.python.org/downloads/)\n"
    "  Windows: winget install Python.Python.%d.%d   (or the python.org installer)\n"
    "  Linux:   your package manager, or pyenv"
) % (RECOMMENDED_PYTHON + RECOMMENDED_PYTHON + RECOMMENDED_PYTHON)

if best is None or (best[0] < MIN_PYTHON and not ALLOW_OLD_PYTHON):
    raise RuntimeError(
        "No Python >= %d.%d was found on this machine.\n%s" % (MIN_PYTHON[0], MIN_PYTHON[1], INSTALL_HELP)
    )

BUILD_PYTHON = best[1]
print("Will build the environment with Python %d.%d (%s)" % (best[0][0], best[0][1], " ".join(BUILD_PYTHON)))
if best[0] < RECOMMENDED_PYTHON:
    print("NOTE: newer than this is recommended (%d.%d).\n%s" % (RECOMMENDED_PYTHON + (INSTALL_HELP,)))

## 2 · Create the virtual environment

In [ ]:
venv_path = ROOT / VENV_DIR
venv_python = venv_path / ("Scripts/python.exe" if IS_WINDOWS else "bin/python")

if RECREATE_VENV and venv_path.exists():
    if Path(sys.prefix).resolve() == venv_path.resolve():
        raise RuntimeError(
            "This notebook is running inside the environment you asked it to rebuild. "
            "Switch the kernel to a Python outside %s and run again." % VENV_DIR
        )
    backup = ROOT / ("%s.old-%d" % (VENV_DIR, int(time.time())))
    shutil.move(str(venv_path), str(backup))
    print("Moved the old environment to %s (safe to delete once the new one works)." % backup.name)

if not venv_path.exists():
    print("Creating %s ..." % VENV_DIR)
    run(BUILD_PYTHON + ["-m", "venv", VENV_DIR])
else:
    print("%s already exists. Set RECREATE_VENV = True to rebuild it." % VENV_DIR)

version_text = run([venv_python, "-c", "import sys; print(sys.version.split()[0])"], show=False).stdout.strip()
print("Environment Python:", version_text)

## 3 · Install the packages

In [ ]:
print("Installing packages (this can take a minute)...")
run([venv_python, "-m", "pip", "install", "--progress-bar", "off", "--no-color", "--disable-pip-version-check", "--upgrade", "pip"], tail=2)
run([venv_python, "-m", "pip", "install", "--progress-bar", "off", "--no-color", "--disable-pip-version-check", "-r", "requirements.txt"], tail=3)

## 4 · Register the Jupyter kernel

In [ ]:
run([
    venv_python, "-m", "ipykernel", "install", "--user",
    "--name", KERNEL_NAME, "--display-name", KERNEL_DISPLAY_NAME,
])

## 5 · Check your `.env` file

In [ ]:
env_file = ROOT / ".env"
example_file = ROOT / ".env.example"

if not env_file.exists():
    if example_file.exists():
        shutil.copy(str(example_file), str(env_file))
        print("Created .env from .env.example.")
    else:
        print("No .env and no .env.example found. Create .env with a line: EIA_API_KEY=your_key")
else:
    print(".env already exists (left untouched).")

value = ""
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.startswith("EIA_API_KEY="):
            value = line.split("=", 1)[1].strip().strip('"').strip("'")

if not value or value == "your_key_here":
    print("ACTION NEEDED: open .env and paste your EIA API key after EIA_API_KEY=  (free: https://www.eia.gov/opendata/)")
else:
    print("EIA_API_KEY is set (%d characters)." % len(value))   # the key itself is never printed

## 6 · Verify

In [ ]:
check = '''
import importlib, sys
print("Python", sys.version.split()[0])
for name in ["pandas", "requests", "dotenv", "ipykernel"]:
    module = importlib.import_module(name)
    print("%-10s %s" % (name, getattr(module, "__version__", "ok")))
'''
run([venv_python, "-c", check])

## Next steps

1. In VS Code, reload the window (**Developer: Reload Window**), open your analysis notebook, and choose the kernel **Python (EIA SEDS)** from the kernel picker (top right).
2. If you haven't yet, add your API key to `.env`.
3. To update packages later, run `01_update_environment.ipynb`. To move to a newer Python, install it, set `RECREATE_VENV = True` above, and run this notebook again (use a kernel *outside* `.venv` for that).